# Figure 1c — fit-recovery yield (3-sigma position + colour) across read noise x peak QE

For a single, isolated emitter already centred in a 2x2-Bayer-unit-cell ROI, bootstrap fits
are run across a 2-D grid of read noise (RMS e-) and peak per-pixel QE, for three real dyes
(ATTO 488, ATTO 565, ATTO 647N). Each bootstrap draws a fresh position (+/-1 px of ROI centre),
photon count, and noise realisation, then re-fits from scratch. Success requires the fit to
recover the true position within `POS_ERR_MULTIPLIER` x its own reported localisation error
(a conventional "3-sigma" bar) **and** the true spectral fingerprint within
`COLOUR_ERR_MULTIPLIER` x its own reported colour error — this is the yield surface shown in
the paper.

**`FAST_MODE` flag (below):** `True` (default) runs a small, fast grid (few read-noise/QE
points, a few hundred bootstraps) that reproduces the same trend in minutes. `FAST_MODE = False`
reproduces the full published sweep (100 read-noise points x 60 QE points x 3 dyes x 20,000
bootstraps each) — this is genuinely hours-to-days of compute; run it only once the FAST_MODE
comparison below has been reviewed.

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(name)s — %(levelname)s — %(message)s',
)

In [ ]:
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Bare `pyS3M.X` imports (not `from src.X import ...`): AnalysisPipeline/SR_Functions add
# `src/` to `sys.path` and bare-import internally, so importing via the `pyS3M.` package path
# keeps every Enum/dataclass identity (e.g. FittingStrategy) consistent with what the fitting
# pipeline itself constructs -- see notebooks/figures/SI/FigureSI_EmitterDensityTests.ipynb.
from pyS3M import IOFunctions, SpectralFunctions, MaskFunctions, PlottingBase, sCMOSFunctions
from pyS3M.simulation.multicolour import (
    MultiC_Sim_Funcs, FittingStrategy, SimulationConfig,
)

IO = IOFunctions.IO_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
MSF = MultiC_Sim_Funcs()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

REPO_ROOT = Path.cwd().resolve().parents[1]  # notebooks/figures -> repo root
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT = Path("outputs") / "Figure_1c"  # git-ignored (bare `outputs/` rule in .gitignore)

## Sweep size: `FAST_MODE` vs. the full published grid

In [ ]:
FAST_MODE = True  # False -> the real, hours-to-days-long full sweep this figure was published with

dyes = ['ATTO 488', 'ATTO 565', 'ATTO 647N']
n_photons = 1000

if FAST_MODE:
    n_bootstrap = 300
    read_noise_space = np.logspace(np.log10(0.1), np.log10(10.0), 6)
    peak_qy_space = np.linspace(0.0, 1.0, 6)
else:
    n_bootstrap = 20000
    read_noise_space = np.logspace(np.log10(0.1), np.log10(10.0), 100)
    peak_qy_space = np.linspace(0.0, 1.0, 60)

n_photon_space = np.array([n_photons], dtype=float)

# Ground-truth position within POS_ERR_MULTIPLIER x the fit's own reported error is the
# success criterion, so no bootstrap-position ROI-bounds check is needed to define "success"
# any more -- but the ROI size is still needed to size the simulated image.
# `test_simulation_method_2d_sweep` resizes to `n_unit_cells` x the camera's 2x2 Bayer unit
# cell when a `mosaic_unit` is set (it is, below), overriding any requested image_size
# directly -- so the ROI size used everywhere else is derived the same way, not hardcoded, to
# avoid the two silently diverging.
n_unit_cells = 7
pixel_size = 69  # nm (Ximea)

sweep_tag = "fast" if FAST_MODE else "full"
save_folder = OUT_ROOT / sweep_tag
save_folder.mkdir(parents=True, exist_ok=True)

print(f"FAST_MODE={FAST_MODE}  n_bootstrap={n_bootstrap}")
print(f"read_noise: {read_noise_space[0]:.4f}-{read_noise_space[-1]:.2f} RMS e-  ({len(read_noise_space)} pts)")
print(f"peak_qy:    {peak_qy_space[0]:.2f}-{peak_qy_space[-1]:.2f}  ({len(peak_qy_space)} pts)")
print(f"Total fits: {len(dyes) * len(read_noise_space) * len(peak_qy_space) * n_bootstrap:,}")
print(f"Saving to:  {save_folder}")

## Camera calibration & spectral setup

In [ ]:
gain = IO.read_tiff(str(CALIBRATION_DIR / "gain.tif"))
offset = IO.read_tiff(str(CALIBRATION_DIR / "offset.tif"))
rqe = IO.read_tiff(str(CALIBRATION_DIR / "rqe.tif"))

gain_median = float(np.median(gain))
offset_median = float(np.median(offset))
rqe_median = float(np.median(rqe))

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])  # B=0, G=1, R=2
base_pixel_QYs = pixel_QYs / np.max(pixel_QYs)  # normalised to peak=1; sweep rescales by peak_qy

masks = M_F.get_masks(size_x=2 * n_unit_cells, size_y=2 * n_unit_cells)
filters = []

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {'sigma': 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = 'image'

print(f"gain={gain_median:.3f}  offset={offset_median:.3f}  rqe={rqe_median:.4f}")
print(f"base_pixel_QYs peak: {np.max(base_pixel_QYs):.4f} (should be 1.0)")

In [ ]:
camera_parameters_base = {
    'gain':                np.full((2 * n_unit_cells, 2 * n_unit_cells), gain_median),
    'offset':              np.full((2 * n_unit_cells, 2 * n_unit_cells), offset_median),
    'variance':            np.full((2 * n_unit_cells, 2 * n_unit_cells), 1.0),   # placeholder, overridden per rn
    'readnoise':           1.0,                                                    # placeholder, overridden per rn
    'rqe':                 np.full((2 * n_unit_cells, 2 * n_unit_cells), rqe_median),
    'masks':               masks,
    'pixel_QYs':           base_pixel_QYs,
    'pixel_order':         ['B', 'G', 'R'],
    'pixel_order_indices': {'B': 0, 'G': 1, 'R': 2},
    'mosaic_unit':         M_F.mosaic_unit,
}

config = SimulationConfig(
    n_bootstrap=n_bootstrap,
    background_photons=5.0,
    background_colour=[1, 1, 1],
    NA=1.49,
    pixel_size=pixel_size,
    cpu_fraction=0.9,
    save_raw_results=True,
    subtractx0y0=False,
    saverawimages=False,
    use_stochastic_photons=True,
    save_summary_csvs=False,
    verbose=False,
    n_unit_cells=n_unit_cells,
)

roi_size = n_unit_cells * M_F.mosaic_unit.shape[0]
print(f"Simulated ROI size actually used by test_simulation_method_2d_sweep: {roi_size}x{roi_size} px")

## Run the 2-D sweep

Factorised across (n_photon, peak_qy): the photoelectron batch is generated once per (n_photon, peak_qy) pair and reused across every read-noise value in the inner loop (Poisson thinning across the QY axis), so this is far cheaper than a naive triple loop.

In [ ]:
start = time.time()
for dye in dyes:
    MSF.test_simulation_method_2d_sweep(
        dye=dye,
        filters=filters,
        wavelength=wavelength,
        camera_parameters=camera_parameters_base,
        save_folder=str(save_folder),
        n_photon_space=n_photon_space,
        smoothing_function=smoothing_function,
        strategy=FittingStrategy.STANDARD_DATA,
        read_noise_space=read_noise_space,
        peak_qy_space=peak_qy_space,
        config=config,
        overwrite=False,  # resume-safe: re-running this cell skips combinations already on disk
    )

print(f"\nAll grid points complete in {(time.time() - start) / 60.0:.2f} min.")

## Fit-recovery yield: position + colour

For every `(dye, peak_qy, read_noise)` grid cell, `yield_recovery_pos_colour` is the fraction
of bootstraps where:

- the ground-truth position lies within `POS_ERR_MULTIPLIER` x the fit's own reported
  localisation error (`xc_err`/`yc_err`) — a conventional "3-sigma" bar, and
- the fitted spectral fingerprint (`A_B`/`A_G`/`A_R`) lies within `COLOUR_ERR_MULTIPLIER` x its
  own fitted error of the dye's *actual per-bootstrap realised* fingerprint
  (`{dye}_nphot_{n}_true_colour_fractions.csv`, saved by `test_simulation_method_2d_sweep` from
  the same stochastic photon draw that generated that bootstrap's image) rather than the single
  population-average fingerprint — these differ bootstrap-to-bootstrap because of finite-photon
  colour-sampling noise. Falls back to the population average for older raw results computed
  before this was saved.

In [ ]:
POS_ERR_MULTIPLIER = 3.0     # x xc_err/yc_err -- conventional "3-sigma" bar, ~99.7% per-axis coverage for a calibrated fit
COLOUR_ERR_MULTIPLIER = 3.0  # x A_*_err, same rationale, applied to the optional colour layer

n_qy, n_rn = len(peak_qy_space), len(read_noise_space)
yield_surf = {dye: np.full((n_qy, n_rn), np.nan) for dye in dyes}

for dye in dyes:
    dyestr = dye.replace(' ', '-').replace('/', '-')

    # Per-bootstrap true colour fraction (the actual finite-photon-sampling colour draw,
    # saved once per (dye, n_photon) at qy_max -- see test_simulation_method_2d_sweep),
    # not the single population-average fingerprint. Falls back to the population
    # average for combinations computed before this was saved -- re-run the sweep cell
    # above to backfill (cheap: existing per-(qy,rn) raw results are still skipped).
    true_colour_path = save_folder / f'{dye}_nphot_{n_photons:.0f}_true_colour_fractions.csv'
    true_colour = pd.read_csv(true_colour_path).to_numpy() if true_colour_path.exists() else None
    if true_colour is None:
        print(f'  [{dye}] no true_colour_fractions.csv yet -- falling back to population-average fingerprint')

    for i, peak_qy in enumerate(peak_qy_space):
        for j, rn in enumerate(read_noise_space):
            flag = f'{dyestr}_qy_{peak_qy:.3f}_readnoise_{rn:.6f}_'

            gt_path = save_folder / f'{flag}LM_method_{dye}_fittesting_input_groundtruthpositions.csv'
            raw_path = save_folder / f'{flag}LM_method_{dye}_rawresults.h5'
            inp_path = save_folder / f'{flag}LM_method_{dye}_fittesting_input_parameters.csv'

            if not (gt_path.exists() and raw_path.exists() and inp_path.exists()):
                continue

            gt = pd.read_csv(gt_path)
            results = pd.read_hdf(raw_path)
            inp = pd.read_csv(inp_path).to_numpy()[0]
            results = results[results['photon_level'] == 0]

            x0 = gt['x0'].to_numpy() / pixel_size
            y0 = gt['y0'].to_numpy() / pixel_size

            converged = ~(results['xc'].isna() | results['yc'].isna())
            err_x = np.abs(results['xc'].to_numpy() - x0)
            err_y = np.abs(results['yc'].to_numpy() - y0)

            recovered_pos = (
                converged
                & (err_x < POS_ERR_MULTIPLIER * results['xc_err'].to_numpy())
                & (err_y < POS_ERR_MULTIPLIER * results['yc_err'].to_numpy())
            )

            dye_BGR = inp[-3:]
            dye_BGR = dye_BGR / np.sum(dye_BGR)
            if true_colour is not None and len(true_colour) == len(results):
                true_frac_b = true_colour  # (n_bootstrap, 3) -- per-bootstrap ground truth, B/G/R
            else:
                true_frac_b = np.tile(dye_BGR, (len(results), 1))
            colour_ok = np.ones(len(results), dtype=bool)
            for c, lbl in enumerate(['A_B', 'A_G', 'A_R']):
                colour_ok &= np.abs(results[lbl].to_numpy() - true_frac_b[:, c]) < COLOUR_ERR_MULTIPLIER * results[f'{lbl}_err'].to_numpy()
            yield_surf[dye][i, j] = (recovered_pos & colour_ok).mean()

    print(f"{dye}: recovery_pos_colour {np.nanmin(yield_surf[dye]):.3f}-{np.nanmax(yield_surf[dye]):.3f}")

print('Done.')

## Real camera survey overlay

`Camera_History.ods` is a hand-curated literature/vendor survey of read noise (RMS e-) and peak QE for sCMOS/CMOS cameras, released 2018 onward, compiled for this figure (Model/Year/RMS/PeakQY/Company/Pixelsize columns). Overlaid as white dots on every surface below so the simulated grid can be read directly against where real hardware actually sits.

In [ ]:
camera_data = pd.read_excel(REPO_ROOT / "notebooks" / "figures" / "Camera_History.ods")
camera_data = camera_data[camera_data['Year'] >= 2018]
print(f"{len(camera_data)} cameras (2018+) loaded for overlay")
camera_data[['Model', 'Year', 'RMS', 'PeakQY', 'Company']]

## Figure: fit-recovery yield surface (3-sigma position + colour)

One panel per dye. The 80% contour is drawn as a heuristic benchmark line, not a derived
accuracy threshold. White dots overlay the real camera survey above so the surface can be
read directly against where real hardware sits.

In [ ]:
cmap_yield = LinearSegmentedColormap.from_list('gar', ['#D2222D', '#FFBF00', '#238823', '#007000'])
fill_levels = np.linspace(0, 100, 11)

fig, axs = plotter.two_column_plot(nrows=1, ncols=len(dyes), width=6.75, height=2.5)
axs = np.atleast_1d(axs)

for col, dye in enumerate(dyes):
    ax = axs[col]
    data = yield_surf[dye] * 100
    image = ax.contourf(
        read_noise_space, peak_qy_space, data,
        levels=fill_levels, cmap=cmap_yield, vmin=0, vmax=100, antialiased=False,
    )
    ax.contour(read_noise_space, peak_qy_space, data, levels=[80], colors='k', linewidths=0.6, linestyles='--')
    ax.set_xscale('log')
    ax.grid(False)
    ax.set_xlim([read_noise_space[0], read_noise_space[-1]])
    ax.set_ylim([0.0, 1])
    ax.tick_params(labelsize=7)
    ax.set_xlabel('read noise / RMS e⁻', fontsize=8)
    if col == 0:
        ax.set_ylabel('peak pixel QY', fontsize=8)
    else:
        ax.set_yticklabels('')
    ax.set_title(dye, fontsize=8)
    if col == len(dyes) - 1:
        cbar = plt.colorbar(image, ax=ax, pad=0.01, cmap=cmap_yield)
        cbar.set_ticks([0, 20, 40, 60, 80, 100])
        cbar.set_label('fit accuracy / %', fontsize=8)
    # SVG-only artefact: contourf bands paint no edge by default, so
    # antialiased=False alone still leaves a thin seam between adjacent bands in
    # vector output (invisible in matplotlib's own PNG export, but real in an SVG
    # viewer -- confirmed by rasterising with Inkscape's renderer). Painting each
    # band's edge in its own fill colour closes the seam.
    image.set_edgecolor('face')
    image.set_linewidth(0.3)
    ax.scatter(
        camera_data['RMS'], camera_data['PeakQY'],
        s=10, lw=0.5, facecolor='white', edgecolors='white', zorder=5,
    )

plt.savefig('Fig_1c.svg', dpi=600, format='svg')
plt.show()